In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 661
PROJECT_YEARS = list(range(2018, 2025))
PM25_THRESHOLD = 25.0
MIN_VALID_PM_HOURS = 18
MIN_VALID_COUNT_HOURS = 24
MAX_FIRE_DISTANCE_KM = 500.0
PREFILTER_DISTANCE_KM = 520.0

# Locate the repository project folder and the Source code/Datasets folder.
# This supports the GitHub structure:
#   repository root/
#     Source code/
#       this notebook
#       Datasets/
#         ntepa_greater_darwin_hourly_2018_2024.csv
#         fire_archive_SV-C2_*.csv
#
# It also supports running VS Code/Jupyter from either the repository root
# or directly from the Source code folder.
possible_project_folders = [
    Path.cwd(),
    Path.cwd() / "Source code",
    Path.cwd().parent,
    Path.cwd().parent / "Source code",
]

project_folder = None
data_folder = None

for folder in possible_project_folders:
    folder = folder.resolve()

    # Preferred repository structure: Source code/Datasets
    candidate_data_folder = folder / "Datasets"
    if (candidate_data_folder / "ntepa_greater_darwin_hourly_2018_2024.csv").exists():
        project_folder = folder
        data_folder = candidate_data_folder
        break

    # Backward-compatible fallback: datasets stored directly beside the notebook.
    if (folder / "ntepa_greater_darwin_hourly_2018_2024.csv").exists():
        project_folder = folder
        data_folder = folder
        break

if project_folder is None or data_folder is None:
    raise FileNotFoundError(
        "The NT EPA dataset could not be found. Expected the project structure "
        "'Source code/Datasets/ntepa_greater_darwin_hourly_2018_2024.csv'. "
        "Run this notebook from the repository root or from the Source code folder."
    )

output_root = project_folder / "outputs_prt661"
table_dir = output_root / "tables"
processed_dir = output_root / "processed"

for folder in [output_root, table_dir, processed_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_folder)
print("Dataset folder:", data_folder)
print("Output folder:", output_root)


Project folder: C:\Users\nguye\Documents\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code
Dataset folder: C:\Users\nguye\Documents\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code\Datasets
Output folder: C:\Users\nguye\Documents\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code\outputs_prt661


In [ ]:
ntepa_file = data_folder / "ntepa_greater_darwin_hourly_2018_2024.csv"
df = pd.read_csv(ntepa_file)
df["datetime_local"] = pd.to_datetime(df["datetime_local"], errors="coerce")
df["date"] = df["datetime_local"].dt.normalize()
df["year"] = df["datetime_local"].dt.year

required_columns = {
    "datetime_local", "station", "latitude", "longitude",
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"NT EPA file is missing required columns: {sorted(missing_columns)}")

quality_columns = [
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
]

overview = pd.DataFrame({
    "metric": [
        "rows", "columns", "date_min", "date_max", "stations",
        "full_row_duplicates", "station_timestamp_duplicates"
    ],
    "value": [
        len(df), df.shape[1], df["datetime_local"].min(), df["datetime_local"].max(),
        ", ".join(sorted(df["station"].dropna().astype(str).unique())),
        int(df.duplicated().sum()),
        int(df.duplicated(["station", "datetime_local"]).sum()),
    ]
})
print("\nRaw NT EPA overview:")
print(overview.to_string(index=False))
overview.to_csv(table_dir / "epa_raw_overview.csv", index=False)

missing_by_year_station = (
    df.groupby(["year", "station"])[quality_columns]
      .agg(lambda s: s.isna().mean() * 100)
      .round(2)
      .reset_index()
)
missing_by_year_station.to_csv(
    table_dir / "epa_missing_pct_year_station_variable.csv", index=False
)

raw_summary = df[quality_columns].describe(
    percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
).T
raw_summary.to_csv(table_dir / "epa_raw_numeric_summary.csv")
print("\nRaw numeric summary:")
print(raw_summary.round(3).to_string())
